# 02 — Veri Ön İşleme

**Amaç:** Data leakage'ı önleyerek doğru bir ön işleme pipeline'ı kurmak.

**Kurallar:**
- `FILENAME`, `URL`, `Domain`, `Title` sütunları DROP edilecek (tanımlayıcı → data leakage)
- `TLD` → LabelEncoder (sadece train'e fit)
- StandardScaler → sadece train'e fit, test'e transform
- Train/Test: %80/%20, stratify=y, random_state=42

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from src.preprocessing import (
    load_data, drop_identifier_columns, encode_tld,
    scale_features, split_data
)

DATA_PATH = '../data/PhiUSIIL_Phishing_URL_Dataset.csv'
FIG_DIR = '../results/figures/'
FIG_KWARGS = dict(dpi=150, bbox_inches='tight')
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style='whitegrid')

## 1. Veri Yükleme

In [ ]:
df = load_data(DATA_PATH)
print(f'Ham veri: {df.shape}')

## 2. Tanımlayıcı Sütunları Sil (Data Leakage Önleme)

In [ ]:
df = drop_identifier_columns(df)
print(f'Temizlenmiş veri: {df.shape}')
print('Kalan sütunlar:')
print(df.columns.tolist())

## 3. Özellik ve Hedef Ayrımı

In [ ]:
y = df['label']
X = df.drop(columns=['label'])
print(f'X: {X.shape}, y: {y.shape}')
print(f'TLD tipi: {X["TLD"].dtype if "TLD" in X.columns else "N/A"}')

## 4. Train/Test Bölünmesi (ÖNCE böl, SONRA encode/scale)

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)

print(f'Train sınıf dağılımı:\n{y_train.value_counts()}')
print(f'Test sınıf dağılımı:\n{y_test.value_counts()}')

## 5. TLD Encoding (sadece train'e fit)

In [ ]:
X_train, tld_encoder = encode_tld(X_train, fit=True)
X_test, _ = encode_tld(X_test, fit=False, encoder=tld_encoder)

print(f'TLD encode sonrası X_train dtype: {X_train["TLD"].dtype if "TLD" in X_train.columns else "N/A"}')
print(f'Benzersiz TLD sayısı: {len(tld_encoder.classes_) if tld_encoder else 0}')

## 6. StandardScaler (sadece train'e fit)

In [ ]:
X_train_sc, X_test_sc, scaler = scale_features(X_train, X_test)

print('Train skalasından örnek (ilk 3 özellik):')
print(pd.DataFrame({
    'mean (should be ~0)': X_train_sc.mean()[:3].round(4).values,
    'std (should be ~1)': X_train_sc.std()[:3].round(4).values,
}, index=X_train_sc.columns[:3]))

## 7. Sonuçları Kaydet (joblib ile)

In [ ]:
# Preprocessed data (numpy arrays daha hızlı)
import joblib

os.makedirs('../results/metrics', exist_ok=True)

joblib.dump({
    'X_train': X_train_sc,
    'X_test': X_test_sc,
    'y_train': y_train,
    'y_test': y_test,
    'feature_names': list(X_train_sc.columns),
    'tld_encoder': tld_encoder,
    'scaler': scaler,
}, '../results/metrics/preprocessed_data.joblib')

print('Ön işlenmiş veri kaydedildi: results/metrics/preprocessed_data.joblib')
print(f'\nÖzellik sayısı: {X_train_sc.shape[1]}')
print(f'Train: {X_train_sc.shape[0]:,} örnek')
print(f'Test:  {X_test_sc.shape[0]:,} örnek')

## Özet

| Adım | Detay |
|------|-------|
| Tanımlayıcı sütunlar | FILENAME, URL, Domain, Title — DROP edildi |
| TLD encoding | LabelEncoder — sadece train'e fit |
| Scaling | StandardScaler — sadece train'e fit |
| Train/Test | %80/%20, stratify, random_state=42 |
| Data leakage | YOK |

Bir sonraki adım: Özellik seçimi (03_feature_selection.ipynb)